# Table 2 / Table 3 / Table 4 / Figure 9 — สคริปต์รวมเดียว (แก้ปัญหาผลไม่ตรงกัน)

**Script file:** `Paper1_SpatialStats_Consolidated.ipynb`

**ทำไมต้องรวมเป็นสคริปต์เดียว:** ก่อนหน้านี้ Table 2 (Moran's I), Table 3 (Gi* hotspots), Table 4 (sensitivity), และ Figure 9 ถูกคำนวณจากคนละสคริปต์ ซึ่งสร้าง spatial weights ต่างวิธีกัน (บางสคริปต์ใช้พิกัดองศาตรงๆ บางสคริปต์แปลงเป็น UTM เมตรก่อน) ทำให้ได้ "เพื่อนบ้านใกล้ที่สุด" คนละชุดสำหรับจุดที่อยู่ชายขอบ cluster และได้ค่า Gi*/Moran's I ไม่ตรงกันระหว่างการรันแต่ละครั้ง (พบ 3 vs 4 vs 0 significant hotspots ในการตรวจสอบก่อนหน้านี้)

**หลักการแก้ไข:**
1. Resolve พิกัด GPS **ครั้งเดียว** แล้วบันทึกเป็นไฟล์ตายตัว (`site_coordinates_locked.csv`) — ใช้ไฟล์นี้ซ้ำในการรันครั้งต่อๆ ไป แทนการ resolve จาก Google Maps link ใหม่ทุกครั้ง
2. แปลงพิกัดเป็น **UTM (EPSG:32647)** ก่อนสร้าง spatial weights ทุกจุด (ตรงตาม methodology ที่ระบุไว้ในต้นฉบับ)
3. คำนวณ Moran's I และ Gi* **ในสคริปต์เดียวกัน จากอ็อบเจ็กต์ weights เดียวกัน** แล้วสร้าง Table 2, Table 3, Table 4, Figure 9 ต่อจากผลลัพธ์ชุดเดียวนั้นทั้งหมด — ไม่มีทางได้ผลไม่ตรงกันอีก เพราะไม่มีการคำนวณซ้ำคนละที่
4. ตั้ง `np.random.seed(12345)` ก่อนเรียก permutation-based functions ทุกตัว เพื่อให้ p-value จาก permutation ทำซ้ำได้ด้วย (ส่วนนี้ไม่ใช่สาเหตุหลักของปัญหา แต่ทำไว้เพื่อความสมบูรณ์)

**หมายเหตุ:** Table 6 (ระยะทางไปถนน/โรงงานปูนของจุด hotspot) ต้องใช้ pipeline คำนวณระยะทางจาก OSM road centerline ที่แยกต่างหาก (Section 2.9/3.6) ซึ่งไม่ได้รวมไว้ในสคริปต์นี้ — เมื่อรันสคริปต์นี้เสร็จแล้ว ให้นำ **รายชื่อ site ที่เป็น significant hotspot รอบนี้** (พิมพ์ไว้ท้าย Section 5) ไปใส่ในสคริปต์ Table 6/Section 3.6 เดิมแทนรายชื่อเก่า

## 1. ติดตั้งไลบรารีและ import

In [ ]:
!pip install -q libpysal esda geopandas matplotlib openpyxl requests

import re, time
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import requests
import openpyxl
import libpysal
from esda.moran import Moran
from esda.getisord import G_Local
from google.colab import files

plt.rcParams['figure.dpi'] = 120
SEED = 12345
print('พร้อมใช้งาน ✅')

## 2. โหลดพิกัด 32 ไซต์

⚠️ **รันครั้งแรก (ยังไม่มีไฟล์ `site_coordinates_locked.csv`): ต้องตั้ง `USE_LOCKED_FILE = False`** (ค่าเริ่มต้นในเซลล์ถัดไปตั้งไว้ถูกต้องแล้ว) แล้วอัปโหลดไฟล์ต้นฉบับ "บันทึกการตรวจวัดปริมาณฝุ่น.xlsx" ใน Section 2A เมื่อกด "Choose Files" ต้อง**เลือกไฟล์และรอให้อัปโหลดเสร็จ 100%** ก่อนปล่อยให้เซลล์ทำงานต่อ ไม่งั้นจะ error

**รันครั้งถัดไป (มีไฟล์ `site_coordinates_locked.csv` จากรอบก่อนแล้ว):** เปลี่ยนเป็น `USE_LOCKED_FILE = True` แล้วข้ามไปอัปโหลดไฟล์ locked ที่ Section 2B แทน จะได้ผลลัพธ์เดิมทุกครั้งรับประกัน

### 2A. Resolve พิกัดจากไฟล์ต้นฉบับ (รันครั้งแรกเท่านั้น)

In [ ]:
USE_LOCKED_FILE = False  # 👉 เปลี่ยนเป็น True ถ้าจะใช้ไฟล์ site_coordinates_locked.csv ที่มีอยู่แล้ว (แล้วข้ามไป Section 2B)

if not USE_LOCKED_FILE:
    print('อัปโหลดไฟล์ "บันทึกการตรวจวัดปริมาณฝุ่น.xlsx": (รอจนกว่าแถบอัปโหลดจะขึ้น 100% ก่อนเซลล์จะทำงานต่อ)')
    uploaded = files.upload()
    if not uploaded:
        raise RuntimeError(
            'ไม่มีไฟล์ถูกอัปโหลด (กด "Choose Files" แล้วไม่ได้เลือกไฟล์ หรือเลือกไม่ทัน) '
            'กรุณารันเซลล์นี้ใหม่อีกครั้ง แล้วเลือกไฟล์ .xlsx ให้เรียบร้อยก่อนปล่อยให้เซลล์ทำงานต่อ'
        )
    RAW_XLSX = list(uploaded.keys())[0]

    wb = openpyxl.load_workbook(RAW_XLSX, data_only=True)
    ws = wb['Sheet1']
    COL_NAME, COL_URL, COL_PM25, COL_TPM, COL_DATE = 0, 1, 6, 9, 13

    records = {}
    order = []
    for row in ws.iter_rows(min_row=3, max_row=1001, values_only=True):
        name = row[COL_NAME]
        if not name:
            continue
        if name not in records:
            records[name] = {'site_id': name, 'url': row[COL_URL], 'pm25': row[COL_PM25], 'tpm': row[COL_TPM]}
            order.append(name)
    sites_raw = [records[n] for n in order]
    print(f'จำนวนจุด: {len(sites_raw)}')

    PATTERNS = [
        re.compile(r"!3d(-?\d+\.\d+)!4d(-?\d+\.\d+)"),
        re.compile(r"@(-?\d+\.\d+),(-?\d+\.\d+)"),
        re.compile(r"[?&]q=(-?\d+\.\d+),(-?\d+\.\d+)"),
        re.compile(r"ll=(-?\d+\.\d+),(-?\d+\.\d+)"),
    ]
    def extract_latlon(url_or_text):
        for pat in PATTERNS:
            m = pat.search(url_or_text)
            if m:
                return float(m.group(1)), float(m.group(2))
        return None, None
    def resolve_link(short_url, timeout=15):
        try:
            r = requests.get(short_url, allow_redirects=True, timeout=timeout, headers={'User-Agent': 'Mozilla/5.0'})
            lat, lon = extract_latlon(r.url)
            if lat is None:
                lat, lon = extract_latlon(r.text[:5000])
            return lat, lon
        except Exception:
            return None, None

    for i, s in enumerate(sites_raw, 1):
        lat, lon = resolve_link(s['url'])
        s['lat'], s['lon'] = lat, lon
        print(f"[{i:02d}/32] {s['site_id']}: {'OK' if lat else 'FAILED'}")
        time.sleep(0.4)

    data = pd.DataFrame(sites_raw).dropna(subset=['lat', 'lon']).reset_index(drop=True)

    # ⚠️ ยืนยันภาคสนามแล้ว: Site 13 (Tab Kwang) อยู่ติดถนนมิตรภาพ ตรงข้าม TPI Polene
    # พิกัดยืนยัน: 14 38 09.6 N, 101 06 55.3 E
    TAB_KWANG_ID = 'จุดตรวจวัดอากาศ-13'
    if TAB_KWANG_ID in data['site_id'].values:
        data.loc[data['site_id'] == TAB_KWANG_ID, 'lat'] = 14 + 38/60 + 9.6/3600
        data.loc[data['site_id'] == TAB_KWANG_ID, 'lon'] = 101 + 6/60 + 55.3/3600
        print(f'✓ ใช้พิกัดยืนยันภาคสนามสำหรับ {TAB_KWANG_ID}')

    data['log_pm25'] = np.log(data['pm25'])
    data[['site_id', 'lat', 'lon', 'pm25', 'log_pm25']].to_csv('site_coordinates_locked.csv', index=False, encoding='utf-8-sig')
    files.download('site_coordinates_locked.csv')
    print('\n✅ บันทึก site_coordinates_locked.csv แล้ว - เก็บไฟล์นี้ไว้ใช้รันครั้งต่อไป (ตั้ง USE_LOCKED_FILE=True)')
    print(data[['site_id', 'lat', 'lon', 'pm25']])

### 2B. หรืออัปโหลดไฟล์ล็อกพิกัดที่มีอยู่แล้ว (ถ้า `USE_LOCKED_FILE = True`)

In [ ]:
if USE_LOCKED_FILE:
    print('อัปโหลดไฟล์ site_coordinates_locked.csv:')
    uploaded_locked = files.upload()
    locked_fname = list(uploaded_locked.keys())[0]
    data = pd.read_csv(locked_fname)
    print(f'โหลดพิกัดที่ล็อกไว้แล้ว: {len(data)} จุด')
    data.head()

## 3. แปลงเป็น UTM (EPSG:32647) และสร้าง Spatial Weights (ใช้ร่วมกันทุกการคำนวณ)

In [ ]:
gdf = gpd.GeoDataFrame(
    data, geometry=gpd.points_from_xy(data['lon'], data['lat']), crs='EPSG:4326'
).to_crs('EPSG:32647')
coords_utm = np.column_stack([gdf.geometry.x, gdf.geometry.y])

pm25 = data['pm25'].values
logpm25 = data['log_pm25'].values if 'log_pm25' in data.columns else np.log(pm25)

# k-NN (k=5) - ใช้ทั้งสำหรับ Moran's I (row-standardized) และ Gi* (binary)
w_knn_r = libpysal.weights.KNN.from_array(coords_utm, k=5); w_knn_r.transform = 'r'
w_knn_b = libpysal.weights.KNN.from_array(coords_utm, k=5); w_knn_b.transform = 'b'

# Distance-band (55 km = 55000 ม.) - ตรงตามที่ระบุใน Section 2.5 ของต้นฉบับ ("~55 km at this latitude")
w_dist_r = libpysal.weights.DistanceBand.from_array(coords_utm, threshold=55000, binary=True); w_dist_r.transform = 'r'
w_dist_b = libpysal.weights.DistanceBand.from_array(coords_utm, threshold=55000, binary=True); w_dist_b.transform = 'b'

print(f'จำนวนจุด: {w_knn_r.n}')
print('สร้าง spatial weights (UTM-based) สำเร็จทั้ง 4 แบบ')

## 4. Table 2 — Global Moran's I (ทุก specification)

In [ ]:
def moran_row(vals, w, dataset_label, weights_label, var_label):
    np.random.seed(SEED)
    m = Moran(vals, w, permutations=999)
    return {'Dataset': dataset_label, 'Weights': weights_label, 'Variable': var_label,
            'I': round(m.I, 4), 'z': round(m.z_sim, 3), 'p': round(m.p_sim, 3)}

mask_excl = data['site_id'] != 'จุดตรวจวัดอากาศ-13'
coords_utm_excl = coords_utm[mask_excl.values]
pm25_excl, logpm25_excl = pm25[mask_excl.values], logpm25[mask_excl.values]
w_knn_excl = libpysal.weights.KNN.from_array(coords_utm_excl, k=5); w_knn_excl.transform = 'r'
w_dist_excl = libpysal.weights.DistanceBand.from_array(coords_utm_excl, threshold=55000, binary=True); w_dist_excl.transform = 'r'

table2_rows = [
    moran_row(pm25, w_knn_r, 'Full (n=32)', 'k-NN (k=5)', 'Raw PM2.5'),
    moran_row(logpm25, w_knn_r, 'Full (n=32)', 'k-NN (k=5)', 'Log PM2.5'),
    moran_row(pm25, w_dist_r, 'Full (n=32)', 'Distance-band', 'Raw PM2.5'),
    moran_row(logpm25, w_dist_r, 'Full (n=32)', 'Distance-band', 'Log PM2.5'),
    moran_row(pm25_excl, w_knn_excl, 'Excl. Tab Kwang (n=31)', 'k-NN (k=5)', 'Raw PM2.5'),
    moran_row(logpm25_excl, w_knn_excl, 'Excl. Tab Kwang (n=31)', 'k-NN (k=5)', 'Log PM2.5'),
    moran_row(pm25_excl, w_dist_excl, 'Excl. Tab Kwang (n=31)', 'Distance-band', 'Raw PM2.5'),
    moran_row(logpm25_excl, w_dist_excl, 'Excl. Tab Kwang (n=31)', 'Distance-band', 'Log PM2.5'),
]
table2 = pd.DataFrame(table2_rows)
table2.to_excel('Table2_morans_i.xlsx', index=False)
table2

## 5. Table 3 / Figure 3 — Getis-Ord Gi* (k-NN, binary, log PM2.5)

In [ ]:
np.random.seed(SEED)
gi_knn = G_Local(logpm25, w_knn_b, transform='B', star=True, permutations=999)

data['gi_knn_z'] = gi_knn.Zs
data['gi_knn_p'] = gi_knn.p_sim
data['sig_hotspot_knn'] = (data['gi_knn_z'] > 1.96) & (data['gi_knn_p'] < 0.05)

n_hotspots = data['sig_hotspot_knn'].sum()
print(f'จำนวน significant hotspots (k-NN, log PM2.5): {n_hotspots}')

table3 = data.loc[data['sig_hotspot_knn'], ['site_id', 'pm25', 'gi_knn_z', 'gi_knn_p']].copy()
table3.columns = ['Site', 'PM2.5 (ug/m3)', 'Gi* z-score', 'p-value']
table3 = table3.sort_values('Gi* z-score', ascending=False).reset_index(drop=True)
table3.to_excel('Table3_hotspots.xlsx', index=False)

print('\n⚠️ เปรียบเทียบกับต้นฉบับ (3 hotspots: Survey-0 z=2.89, Survey-1 z=3.27, Survey-2 z=2.22)')
print('ถ้าจำนวน/รายชื่อไม่ตรง อย่าเพิ่งแก้ต้นฉบับ - ส่งผลลัพธ์นี้กลับมาคุยกันก่อน')
table3

## 6. Table 4 — สรุป Sensitivity (ดึงจากผลลัพธ์ Section 4-5 โดยตรง ไม่คำนวณซ้ำ)

In [ ]:
# Gi* ด้วย distance-band weights (สำหรับเปรียบเทียบ sensitivity ใน Table 4)
np.random.seed(SEED)
gi_dist = G_Local(logpm25, w_dist_b, transform='B', star=True, permutations=999)
data['gi_dist_z'] = gi_dist.Zs
data['gi_dist_p'] = gi_dist.p_sim
data['sig_hotspot_dist'] = (data['gi_dist_z'] > 1.96) & (data['gi_dist_p'] < 0.05)
n_hotspots_dist = data['sig_hotspot_dist'].sum()

# ดึงแถว Moran's I ที่เกี่ยวข้องจาก table2 ที่คำนวณไว้แล้วใน Section 4 (ไม่คำนวณใหม่)
def get_t2(dataset, weights, variable):
    row = table2[(table2.Dataset == dataset) & (table2.Weights == weights) & (table2.Variable == variable)].iloc[0]
    return row['I'], row['p']

I_incl, p_incl = get_t2('Full (n=32)', 'k-NN (k=5)', 'Log PM2.5')
I_excl, p_excl = get_t2('Excl. Tab Kwang (n=31)', 'k-NN (k=5)', 'Log PM2.5')
I_raw, p_raw = get_t2('Full (n=32)', 'k-NN (k=5)', 'Raw PM2.5')
I_log, p_log = get_t2('Full (n=32)', 'k-NN (k=5)', 'Log PM2.5')

table4 = pd.DataFrame([
    {'Sensitivity check': 'Weights matrix (Gi*, full data, log PM2.5)',
     'Specification A': 'k-NN (k=5)', 'Result A': f'{n_hotspots} significant hotspots',
     'Specification B': 'Fixed distance-band (55 km)', 'Result B': f'{n_hotspots_dist} significant hotspots'},
    {'Sensitivity check': "Outlier treatment (Global Moran's I, k-NN, log)",
     'Specification A': 'Including Tab Kwang', 'Result A': f'I={I_incl}, p={p_incl}',
     'Specification B': 'Excluding Tab Kwang', 'Result B': f'I={I_excl}, p={p_excl}'},
    {'Sensitivity check': "Variable transformation (Global Moran's I, k-NN, full data)",
     'Specification A': 'Raw PM2.5', 'Result A': f'I={I_raw}, p={p_raw}',
     'Specification B': 'Log PM2.5', 'Result B': f'I={I_log}, p={p_log}'},
])
table4.to_excel('Table4_sensitivity.xlsx', index=False)
table4

## 7. Figure 9 — Sensitivity map (k-NN vs distance-band) จากผลลัพธ์เดียวกันข้างบน

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
specs = [('gi_knn_z', 'sig_hotspot_knn', f'(a) k-NN (k=5) weights\n{n_hotspots} significant hotspots'),
         ('gi_dist_z', 'sig_hotspot_dist', f'(b) Fixed distance-band (55 km) weights\n{n_hotspots_dist} significant hotspots')]

vmax = max(abs(data['gi_knn_z']).max(), abs(data['gi_dist_z']).max())
norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

for ax, (zcol, sigcol, title) in zip(axes, specs):
    sc = ax.scatter(data['lon'], data['lat'], c=data[zcol], cmap='RdBu_r', norm=norm,
                     s=70, edgecolors='gray', linewidths=0.5, zorder=2)
    hot = data[data[sigcol]]
    ax.scatter(hot['lon'], hot['lat'], facecolors='none', edgecolors='red', s=180, linewidths=2,
               label='Significant hotspot (p<0.05)', zorder=3)
    ax.set_title(title)
    ax.set_xlabel('Longitude (deg E)')
    ax.set_ylabel('Latitude (deg N)')
    ax.legend(fontsize=8)

fig.suptitle('Figure 9. Sensitivity of Gi* hotspot detection to spatial weights specification')
fig.colorbar(sc, ax=axes, label='Gi* z-score (log PM2.5)', shrink=0.8)
plt.savefig('Figure9_sensitivity_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. ดาวน์โหลดผลลัพธ์ทั้งหมด

In [ ]:
data.to_csv('gistar_full_results.csv', index=False, encoding='utf-8-sig')

for f in ['Table2_morans_i.xlsx', 'Table3_hotspots.xlsx', 'Table4_sensitivity.xlsx',
          'Figure9_sensitivity_map.png', 'gistar_full_results.csv']:
    files.download(f)

print('\n=== รายชื่อ site ที่เป็น significant hotspot รอบนี้ (เอาไปใช้ต่อใน Table 6 / Section 3.6) ===')
print(table3['Site'].tolist())

---
✅ **เสร็จสิ้น** — Table 2, Table 3, Table 4, Figure 9 มาจาก spatial weights ชุดเดียวกัน คำนวณในสคริปต์เดียว รับประกันว่าตรงกันเสมอ

⚠️ **ถ้าจำนวน/รายชื่อ hotspot ที่ได้ไม่ตรงกับ 3 จุดเดิมในต้นฉบับ (Survey-0, Survey-1, Survey-2)** นั่นแปลว่าการเปลี่ยนจาก "KNN แบบองศา" เป็น "KNN แบบ UTM เมตร" ทำให้ผลเปลี่ยนจริง (ไม่ใช่ bug) — เป็นการแก้ methodology ให้ถูกต้องขึ้น แต่จะกระทบข้อสรุปหลักของ Discussion 4.1 (ที่บอกว่ามีแต่ purposive site เท่านั้นที่ติดฮอตสปอต) จึงต้องคุยกันก่อนว่าจะปรับเนื้อหาต้นฉบับส่วนไหนเพิ่มเติมไหม อย่าเพิ่งอัปโหลดมาแล้วให้แก้ทันที